In [56]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt
import joblib

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from src.utils_modeling import make_train_val, evaluate_classification

# ----------------- Load data -----------------
DATA_DIR = Path("..") / "data" / "processed"
FILE_PATH = DATA_DIR / "credit_risk_model_ready.csv"

df = pd.read_csv(FILE_PATH)

target_col = "loan_status"  # adjust if needed
df[target_col] = df[target_col].astype(int)

# Simple missing handling
df_baseline = df.dropna().copy()

# ----------------- One-hot encoding (same as before) -----------------
# Separate numeric and non-numeric
num_cols = df_baseline.select_dtypes(include=np.number).columns.tolist()
num_cols = [c for c in num_cols if c != target_col]  # drop target
cat_cols = df_baseline.select_dtypes(exclude=np.number).columns.tolist()

# One-hot encode categorical columns
df_model = pd.get_dummies(df_baseline, columns=cat_cols, drop_first=True)

# ----------------- Train/val split -----------------
X_train, X_val, y_train, y_val = make_train_val(df_model, target_col=target_col)
X_train.shape, X_val.shape

((22910, 22), (5728, 22))

In [57]:
# baseline: Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
results_log_reg = evaluate_classification(log_reg, X_train, y_train, X_val, y_val)

# Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)
results_rf = evaluate_classification(rf, X_train, y_train, X_val, y_val)

# XGBoost
pos_ratio = (y_train == 1).mean()
neg_ratio = (y_train == 0).mean()
scale_pos_weight = neg_ratio / pos_ratio

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42,
)

results_xgb = evaluate_classification(
    xgb_model, X_train, y_train, X_val, y_val
)

/Users/khaledalrashidi/Desktop/Projects/credit_risk_practice/.venv/lib/python3.12/site-packages/sklearn/linear_model/_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Validation Accuracy: 0.8506
Validation F1: 0.5724
Validation AUC: 0.8441

Confusion Matrix:
[[4299  188]
 [ 668  573]]

Classification Report:
              precision    recall  f1-score   support

           0       0.87      0.96      0.91      4487
           1       0.75      0.46      0.57      1241

    accuracy                           0.85      5728
   macro avg       0.81      0.71      0.74      5728
weighted avg       0.84      0.85      0.84      5728

Validation Accuracy: 0.9286
Validation F1: 0.8102
Validation AUC: 0.9276

Confusion Matrix:
[[4446   41]
 [ 368  873]]

Classification Report:
              precision    recall  f1-score   support

           0       0.92      0.99      0.96      4487
           1       0.96      0.70      0.81      1241

    accuracy                           0.93      5728
   macro avg       0.94      0.85      0.88      5728
weighted avg       0.93      0.93      0.92      5728

Validation Accuracy: 0.9043
Validation F1: 0.7822
Validation

In [58]:
# ----------------- (Optional) refit best model on all data -----------------
X_full = df_model.drop(columns=[target_col])
y_full = df_model[target_col]

# Refit XGBoost on full dataset (train + val)
xgb_model.fit(X_full, y_full)

# ----------------- Save artifacts for the app -----------------
feature_names = X_full.columns.tolist()

artifacts = {
    "model": xgb_model,
    "feature_names": feature_names,  # columns AFTER get_dummies
    "cat_cols": cat_cols,           # original categorical columns BEFORE get_dummies
}

MODELS_DIR = ROOT_DIR / "models"
MODELS_DIR.mkdir(exist_ok=True)

model_path = MODELS_DIR / "credit_risk_model.pkl"
joblib.dump(artifacts, model_path)

print(f"Saved model artifacts to {model_path}")

Saved model artifacts to /Users/khaledalrashidi/Desktop/Projects/credit_risk_practice/models/credit_risk_model.pkl


---

In [ ]:
# imports & paths
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import xgboost as xgb
import matplotlib.pyplot as plt

# Add project root (parent of "notebooks") to Python path
ROOT_DIR = Path("..").resolve()
if str(ROOT_DIR) not in sys.path:
    sys.path.append(str(ROOT_DIR))

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

from src.utils_modeling import make_train_val, evaluate_classification

DATA_DIR = Path("..") / "data" / "processed"
FILE_PATH = DATA_DIR / "credit_risk_model_ready.csv"

In [ ]:
# load processed data
df = pd.read_csv(FILE_PATH)
df.head()

In [ ]:
# set target & split
target_col = "loan_status"  # adjust to actual column name once you know it

# If the target is "0"/"1" as strings, convert to int
df[target_col] = df[target_col].astype(int)

In [ ]:
# (Temporary) handle missing values
df_baseline = df.dropna().copy()

In [ ]:
# One-hot encode the categorical columns
# Separate numeric and non-numeric
num_cols = df_baseline.select_dtypes(include=np.number).columns.tolist()
# Don’t treat target as a feature:
num_cols = [c for c in num_cols if c != target_col]

cat_cols = df_baseline.select_dtypes(exclude=np.number).columns.tolist()

# One-hot encode categorical columns
df_model = pd.get_dummies(df_baseline, columns=cat_cols, drop_first=True)

df_model.head()

In [ ]:
X_train, X_val, y_train, y_val = make_train_val(df_model, target_col=target_col)
X_train.shape, X_val.shape

In [ ]:
# baseline: Logistic Regression
log_reg = LogisticRegression(max_iter=1000)
results_log_reg = evaluate_classification(log_reg, X_train, y_train, X_val, y_val)
results_log_reg

In [ ]:
# stronger model: Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=None,
    n_jobs=-1,
    random_state=42,
)
results_rf = evaluate_classification(rf, X_train, y_train, X_val, y_val)
results_rf

In [ ]:
# stronger model: Random Forest or XGBoost

# compute class imbalance ratio on train set
pos_ratio = (y_train == 1).mean()
neg_ratio = (y_train == 0).mean()
scale_pos_weight = neg_ratio / pos_ratio
scale_pos_weight

xgb_model = xgb.XGBClassifier(
    n_estimators=400,
    learning_rate=0.05,
    max_depth=4,
    subsample=0.8,
    colsample_bytree=0.8,
    scale_pos_weight=scale_pos_weight,
    objective="binary:logistic",
    eval_metric="logloss",
    n_jobs=-1,
    random_state=42,
)

results_xgb = evaluate_classification(
    xgb_model, X_train, y_train, X_val, y_val
)
results_xgb

In [ ]:
# Experiment Log
experiment_log = []

experiment_log.append({
    "model": "Logistic Regression",
    "notes": "Baseline, little preprocessing",
    **results_log_reg
})

experiment_log.append({
    "model": "RandomForest",
    "notes": "300 trees, basic params",
    **results_rf
})

experiment_log.append({
    "model": "XGBoost",
    "notes": "400 trees, depth 4, lr 0.05, scale_pos_weight",
    **results_xgb
})

pd.DataFrame(experiment_log)

In [ ]:
# Extract feature importance from the best tree model
feature_names = X_train.columns
importances = xgb_model.feature_importances_

feat_imp = (
    pd.DataFrame({"feature": feature_names, "importance": importances})
    .sort_values("importance", ascending=False)
)

feat_imp.head(20)

In [ ]:

def plot_feature_importance(importances, feature_names, top_n=15):
    idx = np.argsort(importances)[::-1][:top_n]
    top_features = feature_names[idx]
    top_importances = importances[idx]

    plt.figure(figsize=(6, 4))
    plt.barh(range(len(top_features)), top_importances[::-1])
    plt.yticks(range(len(top_features)), top_features[::-1])
    plt.xlabel("Importance")
    plt.title(f"Top {top_n} Feature Importances")
    plt.tight_layout()
    plt.show()

plot_feature_importance(importances, feature_names.values, top_n=15)

In [ ]:
# Default rate by loan grade
df.groupby("loan_grade")["loan_status"].mean().sort_values()

In [ ]:
# Default rate by loan_percent_income bucket
bins = [0, 0.1, 0.2, 0.3, 0.5, 1.0]
labels = ["<10%", "10–20%", "20–30%", "30–50%", ">50%"]
df["burden_bucket"] = pd.cut(df["loan_percent_income"], bins=bins, labels=labels)

df.groupby("burden_bucket")["loan_status"].mean()

In [ ]:
# after training your best model / pipeline
import joblib

joblib.dump(pipeline, "credit_risk_model.pkl")
print("Saved model to credit_risk_model.pkl")